# Week 5 - Milestone 1: Smart Tracking System Blockchain Ledger (Submission)

**Project:** Smart Logistics Tracking System  
**Company:** Kaizen Logistics  
**Blockchain Environment:** Ganache Local Ethereum Testnet  
**Smart Contract:** `IoTDataStorage.sol`

This notebook connects Python to the deployed smart contract, loads the Kaizen Logistics IoT simulation CSV, stores all 100 package records on the blockchain, retrieves the stored ledger records, and validates that the blockchain ledger matches the source CSV.

## 1. Milestone Objective

The objective of this milestone is to demonstrate that simulated IoT logistics data can be securely stored and retrieved from a blockchain ledger. Each package row from the CSV file is stored as one blockchain record using the deployed `IoTDataStorage` smart contract.

The implementation follows this storage format:

- `packageId` = unique package ID from the CSV, such as `PKG001`
- `dataType` = `PackageRecord`
- `dataValue` = full JSON representation of the package row

This structure allows the ledger to preserve the complete package record while keeping the Solidity contract flexible and aligned with the Week 3 smart contract template.

## 2. Import Required Libraries

In [1]:
from pathlib import Path
from dotenv import load_dotenv
from web3 import Web3

import json
import os
import time
import pandas as pd

# Load .env from the project root (same directory as the notebook)
load_dotenv()
print("Environment variables loaded from .env")

Environment variables loaded from .env


## 3. Load the Kaizen Logistics IoT CSV

In [2]:
# Locate the Kaizen Logistics IoT simulation CSV file.
# CSV_PATH can be set in .env; falls back to the default filename search.
env_csv_path = os.getenv("CSV_PATH")
csv_filename = "smart_logistics_tracker_japan_kaizenlogistics.csv"

if env_csv_path:
    candidate_csv_paths = [Path(env_csv_path)]
else:
    candidate_csv_paths = [
        Path("IOT Data Simulation") / csv_filename,
        Path(csv_filename),
        Path("../IOT Data Simulation") / csv_filename
    ]

csv_path = None
for path in candidate_csv_paths:
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        f"{env_csv_path or csv_filename} was not found. "
        "Check CSV_PATH in .env or ensure the file is in 'IOT Data Simulation/'."
    )

df = pd.read_csv(csv_path)

print(f"CSV loaded successfully from: {csv_path}")
print("Dataset shape:", df.shape)
df.head()

CSV loaded successfully from: IOT Data Simulation/smart_logistics_tracker_japan_kaizenlogistics.csv
Dataset shape: (100, 30)


,package_id,tracking_number,timestamp,Origin Location,Origin City,Origin Prefecture,Origin Longitude,Origin Latitude,Order Date,Current Location,...,Delivery City,Delivery Prefecture,Route Distance KM,Estimated Transit Hours,RFID #,RFID Verified,RFID Failure %,RFID Failure Label,RFID Success %,RFID Success Label
0,PKG033,KZJP2026000033,2026-05-04 11:51:30.417739,"Ichihara, Chiba",Ichihara,Chiba,140.1109,35.7272,2026-05-03 10:41:01.319632,"Kofu, Yamanashi",...,Kofu,Yamanashi,129.75,33.54,RFID-KZ-0033,YES,1.98,Low Risk,98.02,Excellent
1,PKG023,KZJP2026000023,2026-05-04 19:54:40.370163,"Ureshino, Saga",Ureshino,Saga,130.3639,33.3656,2026-05-03 07:38:33.276989,"Sakaide, Kagawa",...,Sakaide,Kagawa,354.09,39.08,RFID-KZ-0023,YES,0.91,Low Risk,99.09,Excellent
2,PKG001,KZJP2026000001,2026-05-05 01:20:39.131540,"Yokote, Akita",Yokote,Akita,140.0319,39.8156,2026-05-03 22:25:43.823106,"Minamisoma, Fukushima",...,Minamisoma,Fukushima,219.68,27.88,RFID-KZ-0001,YES,2.66,Low Risk,97.34,Excellent
3,PKG039,KZJP2026000039,2026-05-05 09:19:14.119968,"Kazuno, Akita",Kazuno,Akita,140.1956,39.6428,2026-05-03 10:32:15.317110,"Hida, Gifu",...,Hida,Gifu,547.61,50.35,RFID-KZ-0039,YES,0.92,Low Risk,99.08,Excellent
4,PKG002,KZJP2026000002,2026-05-05 09:27:46.895510,"Etajima, Hiroshima",Etajima,Hiroshima,132.4169,34.2766,2026-05-04 07:17:03.020266,"Transit Scan near Ogori, Fukuoka",...,Ogori,Fukuoka,191.83,45.18,RFID-KZ-0002,YES,2.65,Low Risk,97.35,Excellent


## 4. Validate the Source CSV Before Blockchain Upload

In [3]:
expected_record_count = 100

if len(df) != expected_record_count:
    raise ValueError(f"Expected {expected_record_count} records, but found {len(df)} records.")

if not df["package_id"].is_unique:
    raise ValueError("The package_id column contains duplicate values.")

print("CSV validation passed.")
print("Total package records:", len(df))
print("Unique package IDs:", df["package_id"].nunique())

print("\nDelivery status distribution:")
print(df["Status"].value_counts())

print("\nRFID success summary:")
print(df["RFID Success %"].describe())

CSV validation passed.
Total package records: 100
Unique package IDs: 100

Delivery status distribution:
Status
Delivered        96
In Transit        2
Not Delivered     2
Name: count, dtype: int64

RFID success summary:
count    100.000000
mean      98.140300
std        3.074305
min       75.190000
25%       97.855000
50%       98.530000
75%       99.330000
max       99.840000
Name: RFID Success %, dtype: float64


## 4.1 Worksheet CSV Preview

This section prints the first three CSV records in a readable format for the worksheet.

In [4]:
# Print first three CSV records in worksheet-friendly format
preview_columns = [
    "package_id",
    "tracking_number",
    "timestamp",
    "Origin Location",
    "Origin City",
    "Origin Prefecture",
    "Original Longitude",
    "Origin Latitude",
    "Order Date",
    "Current Location",
    "Estimated Delivery Date",
    "Delivery Exception Reason",
    "Status",
    "Perishable",
    "Temperature",
    "Temperature Issue",
    "Current Longitude",
    "Current Latitude",
    "Delivery Longitude",
    "Delivery Latitude",
    "Delivery City",
    "Delivery Prefecture",
    "Route Distance KM",
    "Estimated Transit Hours",
    "RFID #",
    "RFID Verified",
    "RFID Failure %",
    "RFID Failure Label",
    "RFID Success %",
    "RFID Success Label"
]

available_preview_columns = [col for col in preview_columns if col in df.columns]

for record_number, (_, row) in enumerate(df.head(3).iterrows(), start=1):
    print(f"Record {record_number}")
    for col in available_preview_columns:
        print(f"{col}: {row[col]}")
    print("-" * 60)

Record 1
package_id: PKG033
tracking_number: KZJP2026000033
timestamp: 2026-05-04 11:51:30.417739
Origin Location: Ichihara, Chiba
Origin City: Ichihara
Origin Prefecture: Chiba
Origin Latitude: 35.7272
Order Date: 2026-05-03 10:41:01.319632
Current Location: Kofu, Yamanashi
Estimated Delivery Date: 2026-05-04 20:13:22.439322
Delivery Exception Reason: nan
Status: Delivered
Perishable: NO
Temperature: 25.6
Temperature Issue: Ambient
Current Longitude: 138.6736
Current Latitude: 35.7214
Delivery Longitude: 138.6736
Delivery Latitude: 35.7214
Delivery City: Kofu
Delivery Prefecture: Yamanashi
Route Distance KM: 129.75
Estimated Transit Hours: 33.54
RFID #: RFID-KZ-0033
RFID Verified: YES
RFID Failure %: 1.98
RFID Failure Label: Low Risk
RFID Success %: 98.02
RFID Success Label: Excellent
------------------------------------------------------------
Record 2
package_id: PKG023
tracking_number: KZJP2026000023
timestamp: 2026-05-04 19:54:40.370163
Origin Location: Ureshino, Saga
Origin City:

## 5. Connect Python to Ganache

In [5]:
# Connect to local Ganache blockchain.
# GANACHE_URL is read from .env (default: http://127.0.0.1:7545)
ganache_url = os.getenv("GANACHE_URL", "http://127.0.0.1:7545")

web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("Connected to Ganache successfully.")
    print("Ganache URL:", ganache_url)
    print("Current block number:", web3.eth.block_number)
    print("Available Ganache accounts:", len(web3.eth.accounts))
else:
    raise ConnectionError(f"Connection failed. Ensure Ganache is running at {ganache_url}")

Connected to Ganache successfully.
Ganache URL: http://127.0.0.1:7545
Current block number: 1222
Available Ganache accounts: 10


## 6. Load the Smart Contract ABI and Contract Address

In [6]:
# Deployed smart contract details — loaded from .env
contract_address_raw = os.getenv("CONTRACT_ADDRESS")
if not contract_address_raw:
    raise EnvironmentError("CONTRACT_ADDRESS is not set in .env")

contract_address = web3.to_checksum_address(contract_address_raw)

# ABI_PATH from .env; falls back to candidate search
env_abi_path = os.getenv("ABI_PATH")
if env_abi_path:
    candidate_abi_paths = [Path(env_abi_path)]
else:
    candidate_abi_paths = [
        Path("contracts") / "abi.json",
        Path("abi.json"),
        Path("../contracts") / "abi.json"
    ]

abi_path = None
for path in candidate_abi_paths:
    if path.exists():
        abi_path = path
        break

if abi_path is None:
    raise FileNotFoundError(f"{env_abi_path or 'abi.json'} was not found. Check ABI_PATH in .env")

with open(abi_path, "r") as file:
    abi = json.load(file)

print("ABI loaded successfully from:", abi_path)
print("Contract address:", contract_address)

ABI loaded successfully from: contracts/abi.json
Contract address: 0x85F05208B6C3613f42366dE27BAFBd4df40a8ceb


## 7. Load the Deployed Smart Contract in Python

In [7]:
contract = web3.eth.contract(address=contract_address, abi=abi)

web3.eth.default_account = web3.eth.accounts[0]

print(f"Connected to Smart Contract at {contract_address}")
print("Default sender account:", web3.eth.default_account)

Connected to Smart Contract at 0x85F05208B6C3613f42366dE27BAFBd4df40a8ceb
Default sender account: 0x2c538276DE805CB27b3b41183FA731f65F319Fa1


## 8. Check Existing Blockchain Ledger Count

In [8]:
existing_records = contract.functions.getTotalRecords().call()

print(f"Existing blockchain records before upload: {existing_records}")

if existing_records != 0:
    raise RuntimeError(
        "The deployed contract already contains records. "
        "For a clean Milestone 1 submission, redeploy the smart contract and update contract_address before uploading the 100 CSV records."
    )

BadFunctionCallOutput: Could not transact with/call contract function, is contract deployed correctly and chain synced?

## 9. Prepare CSV Rows for Blockchain Storage

In [ ]:
df_for_chain = df.fillna("").astype(str).copy()

# Preserve the original CSV column order for later decoding and export
original_csv_columns = df_for_chain.columns.tolist()

blockchain_payloads = []

for _, row in df_for_chain.iterrows():
    package_id = row["package_id"]
    data_type = "PackageRecord"
    
    # Convert each full CSV row into JSON while preserving original column order
    data_value = json.dumps(row.to_dict(), ensure_ascii=False)
    
    blockchain_payloads.append({
        "package_id": package_id,
        "data_type": data_type,
        "data_value": data_value
    })

print("Prepared blockchain payloads:", len(blockchain_payloads))
print("\nOriginal CSV column order:")
print(original_csv_columns)

print("\nSample payload:")
print(blockchain_payloads[0])

## 10. Define Function to Send Package Records to the Blockchain

In [ ]:
# Gas limit and write delay read from .env
WRITE_GAS_LIMIT = int(os.getenv("WRITE_GAS_LIMIT", 3000000))
WRITE_DELAY_SECONDS = float(os.getenv("WRITE_DELAY_SECONDS", 0.1))

print(f"Gas limit per transaction: {WRITE_GAS_LIMIT}")
print(f"Delay between writes: {WRITE_DELAY_SECONDS}s")


def send_package_record(package_id, data_type, data_value):
    """
    Sends one package record to the deployed IoTDataStorage smart contract.
    Gas limit is read from WRITE_GAS_LIMIT in .env.
    """
    txn_hash = contract.functions.storeData(
        package_id,
        data_type,
        data_value
    ).transact({
        "from": web3.eth.default_account,
        "gas": WRITE_GAS_LIMIT
    })

    receipt = web3.eth.wait_for_transaction_receipt(txn_hash)

    return {
        "package_id": package_id,
        "transaction_hash": txn_hash.hex(),
        "block_number": receipt.blockNumber,
        "gas_used": receipt.gasUsed,
        "status": receipt.status
    }

## 11. Upload All 100 CSV Records to the Blockchain

In [ ]:
transaction_results = []
total_payloads = len(blockchain_payloads)

for index, payload in enumerate(blockchain_payloads, start=1):
    result = send_package_record(
        payload["package_id"],
        payload["data_type"],
        payload["data_value"]
    )

    transaction_results.append(result)

    print(
        f"{index:03d}/{total_payloads} stored | "
        f"{result['package_id']} | "
        f"Txn: {result['transaction_hash']} | "
        f"Gas used: {result['gas_used']}"
    )

    time.sleep(WRITE_DELAY_SECONDS)

transaction_df = pd.DataFrame(transaction_results)

print("\nUpload completed.")
print("Successful transactions:", transaction_df["status"].sum())
transaction_df.head()

## 11.1 Worksheet Transaction Summary

This section prints the exact values needed for the worksheet after storing all CSV records on the blockchain.

In [ ]:
# Worksheet-ready transaction summary
successful_transactions = int(transaction_df["status"].sum())
first_transaction_hash = transaction_df.iloc[0]["transaction_hash"]
last_transaction_hash = transaction_df.iloc[-1]["transaction_hash"]

print("Number of records successfully stored on blockchain:", successful_transactions)
print("Transaction Hash of first stored record:", first_transaction_hash)
print("Transaction Hash of last stored record:", last_transaction_hash)

## 12. Verify Total Blockchain Records

In [ ]:
total_records = contract.functions.getTotalRecords().call()

print(f"Total IoT records stored on blockchain: {total_records}")

if total_records != expected_record_count:
    raise ValueError(f"Expected {expected_record_count} blockchain records, but found {total_records}.")

print("Blockchain record count verification passed.")

## 13. Retrieve All Blockchain Ledger Records

In [ ]:
retrieved_records = []

for index in range(total_records):
    record = contract.functions.getRecord(index).call()

    retrieved_records.append({
        "blockchain_index": index,
        "blockchain_timestamp": record[0],
        "package_id": record[1],
        "data_type": record[2],
        "data_value": record[3]
    })

ledger_df = pd.DataFrame(retrieved_records)

print("Retrieved blockchain records:", len(ledger_df))
ledger_df.head()

## 14. Decode Retrieved JSON Package Records

In [ ]:
decoded_rows = []

for _, row in ledger_df.iterrows():
    decoded_data = json.loads(row["data_value"])
    
    # Add blockchain metadata after the original CSV fields
    decoded_data["blockchain_index"] = row["blockchain_index"]
    decoded_data["blockchain_timestamp"] = row["blockchain_timestamp"]
    decoded_data["ledger_package_id"] = row["package_id"]
    decoded_data["ledger_data_type"] = row["data_type"]
    
    decoded_rows.append(decoded_data)

decoded_ledger_df = pd.DataFrame(decoded_rows)

# Keep the decoded ledger columns in the same order as the original CSV,
# then append blockchain metadata columns at the end.
metadata_columns = [
    "blockchain_index",
    "blockchain_timestamp",
    "ledger_package_id",
    "ledger_data_type"
]

decoded_ledger_df = decoded_ledger_df[original_csv_columns + metadata_columns]

print("Decoded ledger shape:", decoded_ledger_df.shape)
print("\nDecoded ledger columns:")
print(decoded_ledger_df.columns.tolist())

decoded_ledger_df.head()

## 15. Validate CSV Records Against Blockchain Ledger

In [ ]:
source_compare = df.fillna("").astype(str).copy()

# Compare only the original CSV columns, in the original CSV order
ledger_compare = decoded_ledger_df[original_csv_columns].fillna("").astype(str).copy()

source_compare = source_compare.sort_values("package_id").reset_index(drop=True)
ledger_compare = ledger_compare.sort_values("package_id").reset_index(drop=True)

comparison_result = source_compare.equals(ledger_compare)

print("CSV-to-ledger record match:", comparison_result)

if not comparison_result:
    differences = source_compare.compare(ledger_compare)
    print("Differences found:")
    display(differences)
else:
    print("All 100 CSV records match the retrieved blockchain ledger records.")

## 15.1 Worksheet Retrieved Record Preview

This section prints the first retrieved blockchain record in a readable format for the worksheet.

In [ ]:
# Display the first retrieved blockchain record in worksheet-friendly format
# This uses blockchain_index order to match the first actual stored/retrieved ledger record.
first_ledger_record = decoded_ledger_df.sort_values("blockchain_index").iloc[0]

print("First stored record retrieved from blockchain:")

for col in original_csv_columns:
    print(f"{col}:", first_ledger_record[col])

## 16. Save Retrieved Blockchain Ledger Outputs

In [ ]:
output_dir = Path("IOT Data Simulation")
if not output_dir.exists():
    output_dir = Path(".")

ledger_output_csv = output_dir / "kaizenlogistics_blockchain_ledger_retrieved.csv"
ledger_output_json = output_dir / "kaizenlogistics_blockchain_ledger_retrieved.json"
transaction_output_csv = output_dir / "kaizenlogistics_blockchain_transactions.csv"

decoded_ledger_df.to_csv(ledger_output_csv, index=False)
transaction_df.to_csv(transaction_output_csv, index=False)

with open(ledger_output_json, "w", encoding="utf-8") as file:
    json.dump(decoded_ledger_df.to_dict(orient="records"), file, indent=2, ensure_ascii=False)

print("Retrieved ledger CSV saved to:", ledger_output_csv)
print("Retrieved ledger JSON saved to:", ledger_output_json)
print("Transaction log CSV saved to:", transaction_output_csv)

## 17. Milestone 1 Summary

The notebook successfully connects Python to the local Ganache blockchain and loads the deployed `IoTDataStorage` smart contract using the contract address and ABI from Remix IDE. The Kaizen Logistics IoT dataset is loaded from CSV, converted into blockchain-ready package records, and stored on the blockchain as 100 individual transactions.

After storage, the notebook retrieves all blockchain records, decodes the JSON package payloads, and compares the retrieved ledger data against the original CSV. This confirms that the blockchain ledger contains the same package records as the source IoT simulation dataset.

This Milestone 1 implementation demonstrates how blockchain can support logistics transparency by storing package tracking data in a tamper-resistant ledger.

## 18. Worksheet Completion Checklist

The notebook produces the worksheet-required evidence:

- Ganache connection confirmation
- Smart contract address and ABI loading confirmation
- Total records before storing data
- CSV record count and first three CSV records
- Successful upload of 100 blockchain transactions
- First and last transaction hashes
- Total records stored on blockchain
- First retrieved blockchain record
- CSV-to-ledger validation result